In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ─────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ───────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transpose



Please 

# Longer dataframe

In [2]:
df_deg_bins <- fread("./interm/df_WB_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "30955700 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42489,92.5,27.5,T0DayMin31,2,0,31
42490,92.5,27.5,T0DayMin31,4,0,31


# Computation $$T_{i}^{30}= \sum_{b} \frac{h_{ib}^{30} \times T_b}{24 \times 30}$$

## 30-day exposure

In [3]:
result_30 <- df_deg_bins[, .(WB_i31_60 = sum(N_hours * Temp / 24 / 30, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_30), ncol(result_30))
head(result_30,2)

[1] "126725 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_i31_60
<int>,<dbl>,<dbl>,<dbl>
42489,92.5,27.5,4.406944
42490,92.5,27.5,4.523611


In [4]:
data.frame(result_30) %>% 
        select(WB_i31_60)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic    N    Mean  St. Dev.  Min   Pctl(25) Median Pctl(75)  Max 
----------------------------------------------------------------------
WB_i31_60 126,725 19.92   6.38   -15.69  15.27   21.26   25.21   29.51
----------------------------------------------------------------------


# Categorize temperature into bins

### Wet-bulb temperature thresholds

In [5]:
thresholds_WB <- c(-16,14,16,18,20,22,24,26,28,30)

In [8]:
result_bins <- result_30 %>%
                    mutate(WB_i31_60_bin = cut(WB_i31_60, breaks = thresholds_WB, 
                                           labels = c("<=14°C","(14,16]","(16,18]","(18,20]","(20,22]","(22,24]","(24,26]","(26,28]",">28°C"),
                                           include.lowest = TRUE)                   
                          )
sprintf("%i x %i dataframe", nrow(result_bins), ncol(result_bins))
head(result_bins,2)

[1] "126725 x 5 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_i31_60,WB_i31_60_bin
<int>,<dbl>,<dbl>,<dbl>,<fct>
42489,92.5,27.5,4.406944,<=14°C
42490,92.5,27.5,4.523611,<=14°C


## SAVE

In [9]:
fwrite(result_bins,"./output/df_WB_cells_CDC_31_60.csv")